In [2]:
import sys
import os

# Add parent directory to path to import from app
sys.path.insert(0, os.path.abspath('..'))

from app.db.core.models.market_data_models import *
from app.db.core.models.user_data_models import *
from app.db.core.models.prophit_alts_models import *
from app.db.core.db_config import *
from app.utils.decorators.database import with_session, with_transaction
from app.utils.serialize_output import serialize_sqlalchemy_obj
import json
import pandas as pd
from app.core.calculations.core.helpers import build_returns_df_for_dates
from datetime import datetime, timedelta
import requests
import os
from dotenv import load_dotenv
from sklearn.decomposition import PCA

# Load API key
load_dotenv()
api_key = os.getenv("FMP_API_KEY")

In [3]:
end_date = datetime.now()
start_date = end_date - timedelta(days=365*3)

tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA']

equity_returns_df = build_returns_df_for_dates(
    tickers,
    start_date,
    end_date,
    include_dividends=True,
    drop_rows='any'
)

def get_fmp_data(url):
    """Helper function to fetch data from FMP API"""
    separator = '&' if '?' in url else '?'
    full_url = f"{url}{separator}apikey={api_key}"
    try:
        response = requests.get(full_url)
        response.raise_for_status()
        data = response.json()
        return pd.DataFrame(data) if isinstance(data, list) else pd.DataFrame([data])
    except Exception as e:
        print(f"Error: {e}")
        return pd.DataFrame()

def get_treasury_rates():
    # Treasury Yields - get and rename to desired format
    treasury_data = get_fmp_data("https://financialmodelingprep.com/api/v4/treasury")

    # Rename columns: month1 -> 1m, year1 -> 1y, etc.
    column_mapping = {
        'month1': '1m', 'month3': '3m', 'month6': '6m',
        'year1': '1y', 'year2': '2y', 'year3': '3y', 'year5': '5y', 'year7': '7y', 'year10': '10y', 'year20': '20y', 'year30': '30y'
    }
    treasury_data = treasury_data.rename(columns=column_mapping)

    # Select only the desired columns
    treasury_rates = treasury_data[['date', '1m', '3m', '6m', '1y', '2y', '3y', '5y', '7y', '10y', '20y', '30y']]

    return treasury_rates

rates_df = get_treasury_rates()
print(rates_df)
print(equity_returns_df)


          date    1m    3m    6m    1y    2y    3y    5y    7y   10y   20y  \
0   2025-10-15  4.20  4.03  3.82  3.61  3.50  3.51  3.63  3.82  4.05  4.61   
1   2025-10-14  4.19  4.02  3.80  3.58  3.48  3.47  3.60  3.79  4.03  4.59   
2   2025-10-10  4.19  4.02  3.81  3.60  3.52  3.52  3.65  3.83  4.05  4.60   
3   2025-10-09  4.20  4.03  3.83  3.66  3.60  3.59  3.74  3.92  4.14  4.70   
4   2025-10-08  4.20  4.01  3.82  3.66  3.58  3.58  3.72  3.91  4.13  4.69   
..         ...   ...   ...   ...   ...   ...   ...   ...   ...   ...   ...   
58  2025-07-23  4.37  4.41  4.31  4.08  3.88  3.84  3.94  4.15  4.40  4.93   
59  2025-07-22  4.37  4.41  4.30  4.05  3.83  3.77  3.88  4.09  4.35  4.90   
60  2025-07-21  4.35  4.41  4.30  4.06  3.85  3.81  3.91  4.13  4.38  4.93   
61  2025-07-18  4.35  4.40  4.30  4.08  3.88  3.84  3.96  4.18  4.44  4.99   
62  2025-07-17  4.36  4.41  4.31  4.10  3.91  3.89  4.01  4.22  4.47  5.01   

     30y  
0   4.64  
1   4.62  
2   4.63  
3   4.72  
4   4.72

In [4]:
def calc_spreads(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    df['5y_2y']  = df['5y'] - df['2y']
    return df[['date', '10y_2y', '10y_3m', '5y_2y']]


In [5]:
def calc_volatility(df, window=20):
    yields = df.drop(columns=['date'])
    vol = yields.rolling(window).std()
    vol['date'] = df['date']
    return vol


In [6]:

def calc_pca(df, n_components=3):
    yields = df.drop(columns=['date']).dropna()
    pca = PCA(n_components=n_components)
    pcs = pca.fit_transform(yields - yields.mean())
    pc_df = pd.DataFrame(pcs, columns=[f'PC{i+1}' for i in range(n_components)])
    pc_df['date'] = df.loc[yields.index, 'date'].values
    explained = pca.explained_variance_ratio_
    return pc_df, explained


In [7]:
def calc_correlation(df):
    yields = df.drop(columns=['date'])
    changes = yields.diff().dropna()
    return changes.corr()

In [8]:
def calc_daily_changes(df):
    changes = df.drop(columns=['date']).diff()
    changes['date'] = df['date']
    return changes

In [9]:
def calc_inversion_signals(df):
    df = df.copy()
    df['10y_2y'] = df['10y'] - df['2y']
    df['10y_3m'] = df['10y'] - df['3m']
    
    # Boolean flags
    df['inv_10y2y'] = df['10y_2y'] < 0
    df['inv_10y3m'] = df['10y_3m'] < 0
    
    # Persistence (# of inverted days in a row)
    df['inv_10y2y_streak'] = df['inv_10y2y'].groupby((df['inv_10y2y'] != df['inv_10y2y'].shift()).cumsum()).cumsum()
    df['inv_10y3m_streak'] = df['inv_10y3m'].groupby((df['inv_10y3m'] != df['inv_10y3m'].shift()).cumsum()).cumsum()
    
    # Depth
    avg_depth_2y = df.loc[df['10y_2y'] < 0, '10y_2y'].mean()
    avg_depth_3m = df.loc[df['10y_3m'] < 0, '10y_3m'].mean()
    
    return df[['date','10y_2y','10y_3m','inv_10y2y','inv_10y3m','inv_10y2y_streak','inv_10y3m_streak']], {
        "avg_depth_10y2y": avg_depth_2y,
        "avg_depth_10y3m": avg_depth_3m
    }

In [10]:
def calc_term_premium(df):
    short_end = df[['1y','2y']].mean(axis=1)
    long_end  = df[['20y','30y']].mean(axis=1)
    term_premium = long_end - short_end
    return pd.DataFrame({'date': df['date'], 'term_premium': term_premium})

In [11]:
def get_company_profile(ticker: str):
    """
    Retrieves company profile including beta, volume, and other company stats.
    Returns data like: beta, volAvg (average volume), mktCap, lastDiv, etc.
    """
    url = f"https://financialmodelingprep.com/api/v3/profile/{ticker}"
    return get_fmp_data(url)

def get_quote(ticker: str):
    """
    Retrieves full quote including volume and avgVolume.
    """
    url = f"https://financialmodelingprep.com/api/v3/quote/{ticker}"
    return get_fmp_data(url)

# Example usage
profile_df = get_company_profile('AAPL')
quote_df = get_quote('AAPL')

# Merge on 'symbol'
merged_df = profile_df.merge(quote_df, on='symbol', suffixes=('_profile', '_quote'))

print(merged_df.columns)
print(merged_df[['beta', 'isActivelyTrading', 'isAdr', 'isFund', 'ipoDate', 'earningsAnnouncement', 'sharesOutstanding']])



Index(['symbol', 'price_profile', 'beta', 'volAvg', 'mktCap', 'lastDiv',
       'range', 'changes', 'companyName', 'currency', 'cik', 'isin', 'cusip',
       'exchange_profile', 'exchangeShortName', 'industry', 'website',
       'description', 'ceo', 'sector', 'country', 'fullTimeEmployees', 'phone',
       'address', 'city', 'state', 'zip', 'dcfDiff', 'dcf', 'image', 'ipoDate',
       'defaultImage', 'isEtf', 'isActivelyTrading', 'isAdr', 'isFund', 'name',
       'price_quote', 'changesPercentage', 'change', 'dayLow', 'dayHigh',
       'yearHigh', 'yearLow', 'marketCap', 'priceAvg50', 'priceAvg200',
       'exchange_quote', 'volume', 'avgVolume', 'open', 'previousClose', 'eps',
       'pe', 'earningsAnnouncement', 'sharesOutstanding', 'timestamp'],
      dtype='object')
    beta  isActivelyTrading  isAdr  isFund     ipoDate  \
0  1.094               True  False   False  1980-12-12   

           earningsAnnouncement  sharesOutstanding  
0  2025-10-30T20:00:00.000+0000        148403900

In [12]:
from app.db.core.models.prophit_alts_models import *
from app.db.core.models.market_data_models import *
from app.db.core.db_config import *
from collections import Counter, defaultdict

with MarketSession() as session:
    x = session.query(Ticker).filter(Ticker.sector == 'etf').all()
    industries = set()
    subindustries = []
    industry_to_subindustries = defaultdict(set)

    for ticker in x:
        industries.add(ticker.industry)
        subindustries.append(ticker.sub_industry)
        # Track subindustries for each industry
        industry_to_subindustries[ticker.industry].add(ticker.sub_industry)

    print("Unique industries:")
    for industry in sorted(industries):
        print(industry)

    print("\nUnique subindustries and count of tickers in each:")
    subindustry_counts = Counter(subindustries)
    for subindustry, count in subindustry_counts.most_common():
        print(f"{subindustry}: {count}")

    print("\nNumber of subindustries in each industry:")
    for industry in sorted(industry_to_subindustries):
        print(f"{industry}: {len(industry_to_subindustries[industry])}")

    y = session.query(Ticker).filter(Ticker.sub_industry == 'defense').all()
    for ti in y:
        print(ti.ticker)

Unique industries:
alternative_etfs
commodity_etfs
cryptocurrency_etfs
equity_etfs
fixed_income_etfs

Unique subindustries and count of tickers in each:
sectors: 77
us_factors: 74
emerging_markets: 53
commodities: 30
strategies: 26
developed_countries: 21
preferred_stock: 17
crypto: 14
sovereign: 10
factors: 10
agricultural_commodities: 9
collateralized_loan_obligation: 9
defense: 8
hedge_fund_replication: 8
corporate_bond_etfs: 8
credit: 8
u_s_municipal_bond_etfs: 7
equal_weighted: 6
industrial_commodities: 6
regions: 6
abs_and_mbs: 6
managed_futures: 6
u_s_sector_reits: 5
treasuries: 5
equity_long_short_tactical: 5
risk_parity: 5
interest_rate_and_inflation_hedge: 5
precious_metals: 5
energy: 5
environmental_social_and_corporate_governance: 5
global_dividends: 5
broad_bond_index_etfs: 5
global_real_estate: 4
global_equities: 4
asset_allocation: 4
dividend_strategies: 4
global_regions_dividends: 4
us_major_index: 4
single_country_small_and_mid_caps: 4
commodity_basket: 4
volatility: 3

In [13]:
# ============================================
# ADD A NEW TICKER TO FUND FINAL POSITIONS
# ============================================

import uuid

with ProphitAltsSession() as session:
    # Get the fund you want to add a position to
    fund = session.query(Fund).filter_by(fund_name='consumer_staples_fund').first()
    
    # Create new position with required fields
    new_position = FundFinalPosition(
        fund_id=fund.id,                         # UUID of the fund
        fund_name=fund.fund_name,                # Name of the fund
        ticker_id=uuid.uuid4(),                  # UUID for ticker (generate new or use existing from ticker database)
        ticker_name="NVDA",                      # Ticker symbol
        position=PositionType.LONG,              # LONG or SHORT
        industry="tech",                         # Industry classification
        portfolio_allocation=0.15,               # Allocation as decimal (5% = 0.05)
        reasoning="Strong membership model with pricing power and consistent growth"
    )
    
    session.add(new_position)
    session.commit()
    print(f"✓ Added {new_position.ticker_name} to {new_position.fund_name}")
    print(serialize_sqlalchemy_obj(new_position))


✓ Added NVDA to consumer_staples_fund
{'id': 'cb8fb595-725a-4441-bcd2-b7583b3848dc', 'fund_id': 'e476132e-e862-49ce-a1c9-25424e5626bc', 'fund_name': 'consumer_staples_fund', 'ticker_id': 'dc39c92f-3d8e-4d65-8bbf-a6ca90a74a5f', 'ticker_name': 'NVDA', 'position': <PositionType.LONG: 'LONG'>, 'industry': 'tech', 'portfolio_allocation': '0.15', 'reasoning': 'Strong membership model with pricing power and consistent growth', 'date_created': '2025-10-16T19:21:43.044089', 'date_updated': '2025-10-16T19:21:43.044089'}


In [19]:
with UserSession() as session:
    portfolio = session.query(Portfolio).all()

    for p in portfolio:
        dct = serialize_sqlalchemy_obj(p)
        dct.pop("portfolio_id", None)
        print(dct)

{'id': 33, 'name': 'Portfolio Two', 'ticker': 'AVGO', 'sector': 'equity_sector_information_technology', 'industry': 'semiconductors_and_semiconductor_equipment', 'sub_industry': 'semiconductors', 'allocation': '3.33', 'is_current': False, 'is_discretionary': False, 'supporting_metrics': None, 'reason_for_rec': None, 'created_date': '2025-07-14T11:31:24.291550', 'updated_date': '2025-10-16T14:27:22.431469', 'company_id': '91ad5428-3d79-467f-ae89-83639e20a894', 'user_id': 'f2231c17-92f5-4e78-9d36-a2c0c3f525a5'}
{'id': 34, 'name': 'Portfolio Two', 'ticker': 'RMBS', 'sector': 'equity_sector_information_technology', 'industry': 'semiconductors_and_semiconductor_equipment', 'sub_industry': 'semiconductors', 'allocation': '3.33', 'is_current': False, 'is_discretionary': False, 'supporting_metrics': None, 'reason_for_rec': None, 'created_date': '2025-07-14T11:31:24.331241', 'updated_date': '2025-10-16T14:27:22.431469', 'company_id': '91ad5428-3d79-467f-ae89-83639e20a894', 'user_id': 'f2231c17-